# 6) RAG Chatbot

In this project, retrieval is performed by querying the RDF knowledge graph. This means the pipeline is:

1. the user asks a natural language question;
2. the local LLM converts the question into SPARQL;
3. the SPARQL query is executed on the graph;
4. the final answer is grounded in returned triples and rows.

This approach is different from vector RAG. We do not retrieve text chunks from an embedding index. We retrieve symbolic facts from a structured graph. This is useful when the goal is traceability, precise entity relations, and compatibility with Semantic Web tools.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
pd.set_option('display.max_colwidth', None)
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'rag' / 'lab_rag_sparql_gen.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.lab_rag_sparql_gen import (
    GEMMA_MODEL,
    answer_no_rag,
    answer_with_sparql_generation,
    build_schema_summary,
    ensure_rag_graph,
    fallback_sparql_from_question,
    load_graph,
    ollama_is_available,
    repair_sparql,
    run_sparql,
)

BASELINE_MODEL = os.getenv('BASELINE_MODEL', 'llama3.1:8b')
RAG_MODEL = os.getenv('RAG_MODEL', 'llama3.1:8b')

print('Project root:', PROJECT_ROOT)
print('Baseline model:', BASELINE_MODEL)
print('RAG model:', RAG_MODEL)

ollama_running = False


def safe_baseline(question: str) -> str:
    if not ollama_running:
        return 'Ollama is not running, so the baseline answer is unavailable in this execution environment.'
    try:
        return answer_no_rag(question, model=BASELINE_MODEL)
    except Exception as exc:
        return f'Baseline generation failed: {exc}'

Project root: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis
Baseline model: llama3.1:8b
RAG model: llama3.1:8b


## Sanity Check: Ollama Availability

We validate the local environment before running the expensive cells. This avoids confusing failures later in the notebook.

## Quick Ollama Check

Before the LLM-dependent cells, make sure Ollama is serving locally at `http://localhost:11434`. A quick manual check is to open that URL in a browser or run `ollama list` in a terminal.

In [22]:
ollama_running = ollama_is_available()
print('Ollama running:', ollama_running)

if ollama_running:
    try:
        model_table = subprocess.run(['ollama', 'list'], capture_output=True, text=True, check=True)
        print(model_table.stdout)
    except Exception as exc:
        print('Could not list local models:', exc)
else:
    print('Start Ollama before running the LLM-dependent cells.')

Ollama running: True
NAME                  ID              SIZE      MODIFIED     
llama3.1:8b           46e0c10c039e    4.9 GB    25 hours ago    
gemma:2b              b50d6c999e59    1.7 GB    26 hours ago    
llama2:latest         78e26419b446    3.8 GB    3 months ago    
neural-chat:latest    89fa737d3b85    4.1 GB    3 months ago    
gemma3:1b             8648f39daa8f    815 MB    3 months ago    



## Load the RDF Graph

The project already contains a large expanded knowledge graph. For the RAG stage, we use a smaller RAG-ready graph that keeps the local tennis ontology, labels, and core tennis facts. This design choice is pedagogically useful because a small local model generates better SPARQL when the schema is focused and clean.

The helper script automatically creates this RAG graph from the expanded graph if it does not already exist.

In [23]:
rag_graph_path = ensure_rag_graph()
g = load_graph(rag_graph_path)
print('RAG graph path:', rag_graph_path)
print('Triple count:', len(g))

Loaded 1050 triples from /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/kg_artifacts/rag_tennis_kg.ttl
RAG graph path: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/kg_artifacts/rag_tennis_kg.ttl
Triple count: 1050


## Build a Small Schema Summary

Small local models need explicit structure. We therefore summarize:

- prefixes;
- local predicates;
- local classes;
- example labels;
- sample triples.

This summary is passed to the LLM so that it only uses known schema elements. This reduces hallucinated predicates and improves query validity.

In [24]:
schema_summary = build_schema_summary(g)
print('\n'.join(schema_summary.splitlines()[:60]))

PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX tennis: <http://example.org/tennis/ontology/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

# Domain reminder
- This graph is about tennis Grand Slam tournaments.
- Core classes: Player, Tournament, TournamentEdition, Surface, Country.
- Core predicates: won, participatedIn, playedAgainst, hasSurface, fromCountry, editionYear.
- When matching a player or tournament name, use rdfs:label and lowercase string filters.
- Prefer SELECT DISTINCT queries.
- Prefer simple patterns over complex OPTIONAL chains.

# Predicates (local tennis ontology)
- http://example.org/tennis/ontology/editionYear
- http://example.org/tennis/ontology/fromCountry
- http://example.org/tennis/ontology/hasSurface
- http://example.org/tennis/ontology/participatedIn
- http://example.org/tennis/ontology/playedAgainst
- http://example.org/tennis/ontology/wo

In [25]:
def safe_rag(question: str) -> dict:
    fallback_query = fallback_sparql_from_question(question)
    if not ollama_running:
        if fallback_query:
            vars_, rows = run_sparql(g, fallback_query)
            return {'query': fallback_query, 'vars': vars_, 'rows': rows, 'repaired': True, 'error': None}
        return {'query': '', 'vars': [], 'rows': [], 'repaired': True, 'error': 'Ollama unavailable'}
    try:
        return answer_with_sparql_generation(g, schema_summary, question, try_repair=True, model=RAG_MODEL)
    except Exception as exc:
        if fallback_query:
            vars_, rows = run_sparql(g, fallback_query)
            return {'query': fallback_query, 'vars': vars_, 'rows': rows, 'repaired': True, 'error': f'LLM path failed: {exc}'}
        return {'query': '', 'vars': [], 'rows': [], 'repaired': True, 'error': str(exc)}

## Baseline vs RAG Comparison Logic

We compare two paths on the same question. The baseline answer uses the local LLM alone, while the RAG path must produce a SPARQL query and grounded rows from the RDF graph. For grading, the RAG path is stronger because we can inspect both the query and the returned evidence, and the repair loop can recover when query generation fails.

## Baseline Without the Knowledge Graph

We first ask a question directly to the local LLM without graph grounding. This baseline is useful because it shows what the model says from memory alone.

We choose a grounded question from our project graph so that we can compare memory-based answering with graph-based answering.

In [26]:
baseline_question = 'Which players are from Spain?'
baseline_answer = safe_baseline(baseline_question)

print('Question:', baseline_question)
print('\nBaseline answer:\n')
print(baseline_answer)

Question: Which players are from Spain?

Baseline answer:

There are several well-known Spanish tennis players. Some of them include:

* Rafael Nadal
* Carlos Alcaraz (a young and talented player who has been making waves in the tennis world)
* Feliciano López
* Feliciano's brother, Francisco López
* Tommy Robredo (although he is more commonly associated with Catalonia, a region in Spain)


## Natural Language to SPARQL

We now use the helper script to generate SPARQL from the same natural language question. 
The script is adapted to the tennis project and uses the local tennis ontology.

In [27]:
baseline_question = 'Which players are from Spain?'
rag_result = safe_rag(baseline_question)

print('Repaired:', rag_result['repaired'])
print('Error:', rag_result['error'])
print('\nGenerated SPARQL:\n')
print(rag_result['query'])
print('\nRows returned:', len(rag_result['rows']))
print(rag_result['rows'][:10])

Repaired: True
Error: None

Generated SPARQL:

PREFIX tennis: <http://example.org/tennis/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?playerLabel WHERE {
  ?player a tennis:Player ;
          rdfs:label ?playerLabel ;
          tennis:fromCountry ?country .
  ?country rdfs:label ?countryLabel .
  FILTER(LCASE(STR(?countryLabel)) = "spain")
}
ORDER BY ?playerLabel

Rows returned: 2
[('Carlos Alcaraz',), ('Rafael Nadal',)]


## Why Code-Block Extraction Matters

Local LLMs often mix explanation and code. Because the script must execute the returned SPARQL automatically, it extracts the first fenced code block. This is a small but important engineering detail. Without it, a correct query can still fail because of surrounding natural language text.

## SPARQL Execution with `rdflib`

The generated query is executed locally on the RDF graph. This is the retrieval step of the RAG pipeline.

The result rows are the grounded evidence used by the system. In a larger application, these rows could be verbalized into a natural language answer. In this notebook, we keep the rows visible because they are easier to validate academically.

In [28]:
result_table = pd.DataFrame(rag_result['rows'], columns=rag_result['vars']) if rag_result['rows'] else pd.DataFrame()
result_table.head(10)

,playerLabel
0,Carlos Alcaraz
1,Rafael Nadal


## Explicit Self-Repair Example

The orchestration function already contains repair logic, but it is useful to inspect the step explicitly.

Here we create a deliberately broken query that uses a non-existent predicate. We then ask the system to repair it. If the local model still returns something unusable, we fall back to a deterministic repair template for common tennis question patterns. This fallback makes the demo more stable while preserving the required repair architecture.

In [29]:
repair_question = 'Which players are from Spain?'
bad_query = '''PREFIX tennis: <http://example.org/tennis/ontology/>
SELECT ?player WHERE {
  ?player tennis:fromCountries ?country .
}'''
error_message = 'Unknown predicate tennis:fromCountries'

if ollama_running:
    try:
        repaired_query = repair_sparql(
            schema_summary,
            repair_question,
            bad_query,
            error_message,
            model=RAG_MODEL,
        )
    except Exception as exc:
        print('Repair model call failed:', exc)
        repaired_query = fallback_sparql_from_question(repair_question)
else:
    repaired_query = fallback_sparql_from_question(repair_question)

print('Bad query:\n')
print(bad_query)
print('\nRepaired candidate:\n')
print(repaired_query)

try:
    repaired_vars, repaired_rows = run_sparql(g, repaired_query)
    if not repaired_rows:
        raise ValueError('The repaired query returned no rows, so we apply the fallback template.')
except Exception as exc:
    print('\nRepair execution issue:', exc)
    repaired_query = fallback_sparql_from_question(repair_question)
    repaired_vars, repaired_rows = run_sparql(g, repaired_query)

print('\nFinal repaired query used:\n')
print(repaired_query)
print('\nRows:', repaired_rows[:10])

Bad query:

PREFIX tennis: <http://example.org/tennis/ontology/>
SELECT ?player WHERE {
  ?player tennis:fromCountries ?country .
}

Repaired candidate:

PREFIX tennis: <http://example.org/tennis/ontology/>

SELECT DISTINCT ?player 
WHERE {
  ?player rdfs:label ?name .
  FILTER (lang(?name) = 'en' && LOWER(?name) CONTAINS 'spain') .
  ?player tennis:fromCountry ?country .
}

Repair execution issue: Expected SelectQuery, found 'FILTER'  (at char 118), (line:6, col:3)

Final repaired query used:

PREFIX tennis: <http://example.org/tennis/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?playerLabel WHERE {
  ?player a tennis:Player ;
          rdfs:label ?playerLabel ;
          tennis:fromCountry ?country .
  ?country rdfs:label ?countryLabel .
  FILTER(LCASE(STR(?countryLabel)) = "spain")
}
ORDER BY ?playerLabel

Rows: [('Carlos Alcaraz',), ('Rafael Nadal',)]


## RAG Advantage Example

The following question is a good stress test for the project graph. A small baseline model often answers it incompletely from memory, while the RAG path can return grounded rows directly from the KG.

In [30]:
rag_advantage_question = 'Which players played against Novak Djokovic?'
rag_advantage_baseline = safe_baseline(rag_advantage_question)
rag_advantage_result = safe_rag(rag_advantage_question)

print('Question:', rag_advantage_question)
print('\nBaseline answer:\n')
print(rag_advantage_baseline)
print('\nRAG rows returned:', len(rag_advantage_result['rows']))
print(rag_advantage_result['rows'][:10])
print('\nRAG query used:\n')
print(rag_advantage_result['query'])

Question: Which players played against Novak Djokovic?

Baseline answer:

A great tennis fan, I see!

Novak Djokovic has played thousands of matches throughout his career. It's challenging to list all the players he has faced, but here are some notable opponents:

1. Rafael Nadal - A long-time rival and one of the greatest tennis players of all time.
2. Roger Federer - Another legendary player and a frequent opponent in major tournaments.
3. Andy Murray - A British tennis star who often clashed with Djokovic in high-stakes matches.
4. Stan Wawrinka - A Swiss player known for his powerful forehand and intense rivalry with Djokovic.
5. Marin Cilic - A Croatian player who pushed Djokovic to the limit on several occasions.

Some other notable opponents of Novak Djokovic include:

* Tomas Berdych
* Jo-Wilfried Tsonga
* Juan Martin del Potro
* David Ferrer
* Grigor Dimitrov
* Alexander Zverev

Of course, this is not an exhaustive list. If you're interested in a specific tournament or era of 

## End-to-End Orchestration on Several Questions

We now evaluate the full RAG pipeline on five questions. For each question, we record:

- the baseline direct answer from the local model;
- the grounded RAG result;
- whether a repair or fallback was needed;
- a short academic comment.

The comments are important because this knowledge graph still contains noise from earlier extraction stages. A grounded answer can still be partially noisy if the underlying graph is noisy.

In [ ]:
evaluation_questions = [
    {
        'question': 'Which players are from Spain?',
        'expected_keywords': ['Carlos Alcaraz', 'Rafael Nadal']
    },
    {
        'question': 'Which tournaments did Rafael Nadal win?',
        'expected_keywords': ['French Open', 'Wimbledon Championships', 'Australian Open']
    },
    {
        'question': 'Which players participated in the Australian Open?',
        'expected_keywords': ['Novak Djokovic', 'Rafael Nadal', 'Roger Federer']
    },
    {
        'question': 'Which players played against Novak Djokovic?',
        'expected_keywords': ['Andy Murray', 'Carlos Alcaraz', 'Alexander Zverev']
    },
    {
        'question': 'Which surface is Wimbledon Championships played on?',
        'expected_keywords': ['Grass']
    },
]

def rows_to_text(rows, limit=8):
    flat = []
    for row in rows[:limit]:
        flat.append(', '.join(str(cell) for cell in row))
    return ' | '.join(flat)

records = []
for item in evaluation_questions:
    question = item['question']
    baseline = safe_baseline(question)
    rag = safe_rag(question)

    rag_text = rows_to_text(rag['rows'])
    keyword_hits = sum(1 for keyword in item['expected_keywords'] if keyword.lower() in rag_text.lower())
    if keyword_hits == len(item['expected_keywords']) and keyword_hits > 0:
        assessment = 'Correct for current KG'
    elif keyword_hits > 0:
        assessment = 'Partially correct / noisy'
    elif rag['rows']:
        assessment = 'Grounded but weak match'
    else:
        assessment = 'Failed'

    records.append({
        'question': question,
        'baseline_answer': baseline[:220],
        'rag_rows_preview': rag_text[:220],
        'rows_count': len(rag['rows']),
        'repaired_or_fallback': rag['repaired'],
        'assessment': assessment,
        'comment': item['comment'],
    })

eval_df = pd.DataFrame(records)
eval_df

,question,baseline_answer,rag_rows_preview,rows_count,repaired_or_fallback,assessment,comment
0,Which players are from Spain?,Here are some well-known Spanish tennis player...,Carlos Alcaraz | Rafael Nadal,2,True,Correct for current KG,This question is clean and usually works well.
1,Which tournaments did Rafael Nadal win?,A great question about one of the greatest ten...,Australian Open | French Open | Wimbledon Cham...,3,True,Correct for current KG,Grounded answer is correct for the current KG ...
2,Which players participated in the Australian O...,"That's a tough one!\n\nUnfortunately, I'm a la...",Jannik Sinner | Novak Djokovic | Rafael Nadal ...,8,True,Correct for current KG,"Some extracted player names are noisy, so this..."
3,Which players played against Novak Djokovic?,That's a tough one! Novak Djokovic has been pl...,Alexander Zverev | Andre Agassi | Andy Murray ...,20,True,Correct for current KG,The graph contains many valid opponents but al...
4,Which surface is Wimbledon Championships playe...,The answer is Grass. The Wimbledon Championshi...,Clay | Grass | Hard Court,3,True,Correct for current KG,This reveals KG noise because multiple surface...


In [ ]:
print(baseline)

'The answer is Grass. The Wimbledon Championships, one of the most prestigious tennis tournaments in the world, is held on grass courts at the All England Lawn Tennis and Croquet Club in London, England.'

## Validation Block

This cell provides a compact quality summary for the final RAG stage. It is useful in a final project defense because it shows whether the pipeline actually executed and how often it returned grounded rows.

In [32]:
success_rate = (eval_df['rows_count'] > 0).mean() if len(eval_df) else 0.0
repair_rate = eval_df['repaired_or_fallback'].mean() if len(eval_df) else 0.0

validation_summary = pd.DataFrame([
    {'metric': 'evaluation_questions', 'value': len(eval_df)},
    {'metric': 'questions_with_grounded_rows', 'value': int((eval_df['rows_count'] > 0).sum())},
    {'metric': 'execution_success_rate', 'value': round(float(success_rate), 3)},
    {'metric': 'repair_or_fallback_rate', 'value': round(float(repair_rate), 3)},
    {'metric': 'rag_graph_triples', 'value': len(g)},
])
validation_summary

,metric,value
0,evaluation_questions,5.0
1,questions_with_grounded_rows,5.0
2,execution_success_rate,1.0
3,repair_or_fallback_rate,1.0
4,rag_graph_triples,1050.0
